# Table 2 | PPSA snow pit comparison

Both model columns use the PPSA soil profiles, matching Figure 5. The
[all-pit comparison](../figures/notebooks/figuresX_all_snowpits.ipynb) selects
the cold and warm examples for agreement with both simulations. The table uses
the selected model timestamps and integrates the model layers above ground.
Observed depth, bulk density, and SWE come from the campaign summary. Observed
mean temperature is integrated between measured heights. Model mean
temperature covers the full simulated snow column.

In [4]:
from datetime import datetime
from pathlib import Path
import pandas as pd
import numpy as np

model_output = Path("/home/cassie/python/models/run_snowpack/sites/ppsa/output")

# =============================================================================
# USER SETTINGS
# =============================================================================

pro_paths = {
    "HRRR-AK Model": model_output / "hrrrak_lapserate_ppsa_WY2020-WY2025_soil.pro",
    "Met Mixed Model": model_output / "met_hrrrak_lapserate_ppsa_WY2020-WY2025_soil.pro",
}

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'figures/data/ppsa_snowpit_selection_WY2025.csv').is_file())
selected = pd.read_csv(root / 'figures/data/ppsa_snowpit_selection_WY2025.csv',
                       parse_dates=['date', 'model_time'])
field = pd.read_csv('/hdd/snow_hydrology/field_data/snow_pit_swe_measurements_WY2025.csv',
                    index_col=0, parse_dates=True).loc[selected['date']]
pit_table = pd.DataFrame({
    'pit_name': selected['pit_class'].str.capitalize().map(lambda name: f'{name} pit'),
    'pit_time': selected['model_time'],
    'obs_HS_cm': field['snow_depth_cm'].to_numpy(),
    'obs_bulk_density_kg_m3': field['average_density'].to_numpy(),
    'obs_mean_temperature_C': selected['observed_mean_temperature_c'],
    'obs_SWE_mm': field['swe_mm'].to_numpy(),
})

# =============================================================================
# PARSE SNOWPACK .PRO FILE
# =============================================================================

def parse_snowpack_pro_profiles(path, start, end, codes=("0501", "0502", "0503")):
    profiles = {}
    current_time = None
    current = None
    in_data = False

    def save_profile():
        if current_time is not None and current is not None:
            profiles[current_time] = current

    with Path(path).open("r", errors="replace") as f:
        for line in f:
            line = line.strip()

            if line == "[DATA]":
                in_data = True
                continue

            if not in_data or not line:
                continue

            code = line[:4]

            if code == "0500":
                save_profile()
                current_time = datetime.strptime(line.split(",", 1)[1], "%d.%m.%Y %H:%M:%S")
                if current_time > end:
                    current = None
                    break
                current = {} if current_time >= start else None
                continue

            if code in codes and current is not None:
                parts = line.split(",")
                n = int(float(parts[1]))
                values = np.array([float(x) for x in parts[2:] if x != ""], dtype=float)
                if len(values) != n:
                    raise ValueError(f"{path}: incorrect count for {current_time} field {code}")
                current[code] = values

    save_profile()
    return profiles


def profile_bulk_metrics(profile):
    heights_cm = np.asarray(profile.get("0501", []), dtype=float)
    density = np.asarray(profile.get("0502", []), dtype=float)
    temperature = np.asarray(profile.get("0503", []), dtype=float)

    if len(heights_cm) != len(density) + 1 or len(density) != len(temperature):
        raise ValueError("Expected one more layer boundary than layer values in the soil profile.")

    # 0501 lists the bottom boundary followed by each layer top.
    bottoms, tops = heights_cm[:-1], heights_cm[1:]
    snow = tops > 0
    dz_cm = tops[snow] - np.maximum(bottoms[snow], 0)
    density = density[snow]
    temperature = temperature[snow]
    if not len(dz_cm) or np.any(dz_cm <= 0):
        raise ValueError("Expected ordered snow layers above the soil surface.")
    if not (np.isfinite(density).all() and np.isfinite(temperature).all()
            and (density > 0).all() and (temperature > -100).all()):
        raise ValueError("Invalid density or temperature in a snow layer.")

    dz_m = dz_cm / 100.0

    HS_cm = np.sum(dz_cm)
    SWE_mm = np.sum(density * dz_m)
    bulk_density = SWE_mm / (HS_cm / 100.0)
    mean_temperature = np.average(temperature, weights=dz_cm)

    return {
        "HS_cm": HS_cm,
        "bulk_density_kg_m3": bulk_density,
        "mean_temperature_C": mean_temperature,
        "SWE_mm": SWE_mm,
    }


def get_nearest_profile_metrics(profiles, target_time, tolerance="24h"):
    target_time = pd.to_datetime(target_time)
    times = pd.DatetimeIndex(sorted(profiles))
    if times.empty:
        raise ValueError(f"No model profiles found near {target_time}.")

    idx = times.get_indexer([target_time], method="nearest")[0]
    profile_time = times[idx]
    offset = abs(profile_time - target_time)

    if offset > pd.Timedelta(tolerance):
        raise ValueError(
            f"Nearest profile to {target_time} is {profile_time}, "
            f"outside tolerance {tolerance}"
        )

    metrics = profile_bulk_metrics(profiles[profile_time])
    metrics["profile_time"] = profile_time
    return metrics


# =============================================================================
# EXTRACT METRICS FROM BOTH MODEL FILES
# =============================================================================

pit_times = pd.to_datetime(pit_table["pit_time"])
tolerance = pd.Timedelta(hours=24)
profiles_by_model = {
    name: parse_snowpack_pro_profiles(path, pit_times.min() - tolerance, pit_times.max() + tolerance)
    for name, path in pro_paths.items()
}
rows = []

for _, pit in pit_table.iterrows():
    row = {
        "pit_name": pit["pit_name"],
        "pit_time": pd.to_datetime(pit["pit_time"]),
        "obs_HS_cm": pit["obs_HS_cm"],
        "obs_bulk_density_kg_m3": pit["obs_bulk_density_kg_m3"],
        "obs_mean_temperature_C": pit["obs_mean_temperature_C"],
        "obs_SWE_mm": pit["obs_SWE_mm"],
    }

    for model_name, profiles in profiles_by_model.items():
        metrics = get_nearest_profile_metrics(
            profiles,
            pit["pit_time"],
            tolerance="24h"
        )

        prefix = (
            model_name
            .replace(" ", "_")
            .replace("-", "")
            .replace("/", "_")
        )

        row[f"{prefix}_HS_cm"] = metrics["HS_cm"]
        row[f"{prefix}_bulk_density_kg_m3"] = metrics["bulk_density_kg_m3"]
        row[f"{prefix}_mean_temperature_C"] = metrics["mean_temperature_C"]
        row[f"{prefix}_SWE_mm"] = metrics["SWE_mm"]
        row[f"{prefix}_profile_time"] = metrics["profile_time"]

    rows.append(row)

comparison = pd.DataFrame(rows)

# =============================================================================
# BUILD TABLE LIKE SCREENSHOT
# =============================================================================

model_cols = {
    "HRRR-AK Model": "HRRRAK_Model",
    "Met Mixed Model": "Met_Mixed_Model",
}

table_rows = []

for _, row in comparison.iterrows():

    # Main data row
    table_rows.append({
        ("Date", ""): row["pit_name"] + "\n" + row["pit_time"].strftime("%-d %b\n%Y"),

        ("Snow Depth\n(cm)", "Obs"): row["obs_HS_cm"],
        ("Snow Depth\n(cm)", "HRRR-AK\nModel"): row["HRRRAK_Model_HS_cm"],
        ("Snow Depth\n(cm)", "Met\nMixed\nModel"): row["Met_Mixed_Model_HS_cm"],

        ("Bulk Snow Density\n(kg m$^{-3}$)", "Obs"): row["obs_bulk_density_kg_m3"],
        ("Bulk Snow Density\n(kg m$^{-3}$)", "HRRR-AK\nModel"): row["HRRRAK_Model_bulk_density_kg_m3"],
        ("Bulk Snow Density\n(kg m$^{-3}$)", "Met\nMixed\nModel"): row["Met_Mixed_Model_bulk_density_kg_m3"],

        ("Mean Temperature\n(°C)", "Obs"): row["obs_mean_temperature_C"],
        ("Mean Temperature\n(°C)", "HRRR-AK\nModel"): row["HRRRAK_Model_mean_temperature_C"],
        ("Mean Temperature\n(°C)", "Met\nMixed\nModel"): row["Met_Mixed_Model_mean_temperature_C"],

        ("SWE\n(mm)(kg m$^{-2}$)", "Obs"): row["obs_SWE_mm"],
        ("SWE\n(mm)(kg m$^{-2}$)", "HRRR-AK\nModel"): row["HRRRAK_Model_SWE_mm"],
        ("SWE\n(mm)(kg m$^{-2}$)", "Met\nMixed\nModel"): row["Met_Mixed_Model_SWE_mm"],
    })

    # Error row
    table_rows.append({
        ("Date", ""): "Model -\nObs\nError (%)",

        ("Snow Depth\n(cm)", "Obs"): "",
        ("Snow Depth\n(cm)", "HRRR-AK\nModel"): 100 * (row["HRRRAK_Model_HS_cm"] - row["obs_HS_cm"]) / row["obs_HS_cm"],
        ("Snow Depth\n(cm)", "Met\nMixed\nModel"): 100 * (row["Met_Mixed_Model_HS_cm"] - row["obs_HS_cm"]) / row["obs_HS_cm"],

        ("Bulk Snow Density\n(kg m$^{-3}$)", "Obs"): "",
        ("Bulk Snow Density\n(kg m$^{-3}$)", "HRRR-AK\nModel"): 100 * (row["HRRRAK_Model_bulk_density_kg_m3"] - row["obs_bulk_density_kg_m3"]) / row["obs_bulk_density_kg_m3"],
        ("Bulk Snow Density\n(kg m$^{-3}$)", "Met\nMixed\nModel"): 100 * (row["Met_Mixed_Model_bulk_density_kg_m3"] - row["obs_bulk_density_kg_m3"]) / row["obs_bulk_density_kg_m3"],

        ("Mean Temperature\n(°C)", "Obs"): "",
        ("Mean Temperature\n(°C)", "HRRR-AK\nModel"): "",
        ("Mean Temperature\n(°C)", "Met\nMixed\nModel"): "",

        ("SWE\n(mm)(kg m$^{-2}$)", "Obs"): "",
        ("SWE\n(mm)(kg m$^{-2}$)", "HRRR-AK\nModel"): 100 * (row["HRRRAK_Model_SWE_mm"] - row["obs_SWE_mm"]) / row["obs_SWE_mm"],
        ("SWE\n(mm)(kg m$^{-2}$)", "Met\nMixed\nModel"): 100 * (row["Met_Mixed_Model_SWE_mm"] - row["obs_SWE_mm"]) / row["obs_SWE_mm"],
    })

table = pd.DataFrame(table_rows)
table.columns = pd.MultiIndex.from_tuples(table.columns)

# =============================================================================
# ROUNDING
# =============================================================================

table_display = table.copy()

snow_cols = [col for col in table_display.columns if "Snow Depth" in col[0]]
density_cols = [col for col in table_display.columns if "Density" in col[0]]
swe_cols = [col for col in table_display.columns if "SWE" in col[0]]
temp_cols = [col for col in table_display.columns if "Temperature" in col[0]]

error_rows = table_display[("Date", "")].astype(str).str.contains("Error", na=False)

def format_value(x, decimals=0):
    if pd.isna(x) or x == "":
        return ""
    return f"{float(x):.{decimals}f}"

# Whole numbers for snow depth, density, and SWE data rows
for col in snow_cols + density_cols + swe_cols:
    table_display[col] = table_display[col].apply(lambda x: format_value(x, decimals=0))

# One decimal for temperature data rows
for col in temp_cols:
    table_display[col] = table_display[col].apply(lambda x: format_value(x, decimals=1))

# Percent error rows: one decimal, but leave Obs columns blank
for col in snow_cols + density_cols + swe_cols:
    if col[1] == "Obs":
        table_display.loc[error_rows, col] = ""
    else:
        table_display.loc[error_rows, col] = table.loc[error_rows, col].apply(
            lambda x: format_value(x, decimals=1)
        )

# Leave temperature error row blank
for col in temp_cols:
    table_display.loc[error_rows, col] = ""

## Observed and simulated snow properties

Each pit has its own table. Percentage errors are model minus observation,
divided by the observation. A dash in the temperature error columns indicates that a Celsius percentage error is undefined.

In [5]:
metric_labels = {
    'Snow Depth\n(cm)': 'Snow depth (cm)',
    'Bulk Snow Density\n(kg m$^{-3}$)': 'Bulk snow density (kg/m³)',
    'Mean Temperature\n(°C)': 'Mean snow temperature (°C)',
    'SWE\n(mm)(kg m$^{-2}$)': 'Snow water equivalent (mm)',
}

for index, pit in comparison.iterrows():
    values = table_display.iloc[2 * index]
    errors = table_display.iloc[2 * index + 1]

    def percent_error(metric, source):
        value = errors[(metric, source)]
        return f'{float(value):+.1f}%' if value else '—'

    pit_display = pd.DataFrame([
        {
            'Metric': label,
            'Observed': values[(metric, 'Obs')],
            'HRRR-AK': values[(metric, 'HRRR-AK\nModel')],
            'Met mixed': values[(metric, 'Met\nMixed\nModel')],
            'HRRR-AK error': percent_error(metric, 'HRRR-AK\nModel'),
            'Met mixed error': percent_error(metric, 'Met\nMixed\nModel'),
        }
        for metric, label in metric_labels.items()
    ])
    date = pit['pit_time'].strftime('%-d %b %Y')
    display(
        pit_display.style.hide(axis='index').set_caption(f"{pit['pit_name']} | {date}")
        .set_properties(subset=pit_display.columns[1:], **{'text-align': 'right'})
        .set_table_styles([
            {'selector': 'caption', 'props': [('text-align', 'left'), ('font-weight', 'bold')]},
            {'selector': 'th', 'props': [('text-align', 'left')]},
            {'selector': 'th, td', 'props': [('padding', '5px 9px')]},
        ])
    )

Metric,Observed,HRRR-AK,Met mixed,HRRR-AK error,Met mixed error
Snow depth (cm),130,138,131,+6.5%,+1.0%
Bulk snow density (kg/m³),334,368,353,+10.1%,+5.6%
Mean snow temperature (°C),-3.5,-3.6,-4.2,—,—
Snow water equivalent (mm),434,509,463,+17.3%,+6.7%


Metric,Observed,HRRR-AK,Met mixed,HRRR-AK error,Met mixed error
Snow depth (cm),147,178,155,+21.0%,+5.6%
Bulk snow density (kg/m³),373,351,380,-5.9%,+1.8%
Mean snow temperature (°C),0.1,-0.1,0.0,—,—
Snow water equivalent (mm),548,625,589,+14.0%,+7.5%
